# Generate Canonical Dataset (Gemini 2.5 Flash)

Processes `final_dataset.jsonl` line-by-line, calls **Gemini 2.5 Flash** (free tier via Google AI Studio) for semantic extraction, and deterministically assembles `canonical_dataset_gemini.jsonl`.

**Pipeline:** Read JSONL → Concurrent LLM Extraction → Assembly → Write JSONL

> **Resumable:** The main loop detects already-processed rows and resumes from where it left off.

> **Rate Limits:** Gemini 2.5 Flash free tier allows **15 RPM** and **1M TPM**. The pipeline respects this with a per-request throttle.

In [1]:
import json
import re
import math
import uuid
import time
import os
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
from tqdm.notebook import tqdm
from typing import Dict, Any, Optional, List
from dotenv import load_dotenv

load_dotenv()

try:
    from google import genai
    from google.genai import types
except ImportError:
    raise ImportError(
        "google-genai package not found. Install it with:\n"
        "  pip install google-genai"
    )


In [8]:

# ── Configuration ──
INPUT_PATH  = "/home/abhishek/Desktop/projects/RAG_MOLE/data/final_dataset.jsonl"
OUTPUT_PATH = "/home/abhishek/Desktop/projects/RAG_MOLE/data/processed/canonical_dataset_gemini.jsonl"
MODEL_NAME  = "gemini-3.1-flash-lite"
MAX_WORKERS = 2           # Keep low to stay within free-tier RPM limits
TEMPERATURE = 0.6
MAX_TOKENS  = 2048
TIMEOUT     = 120         # seconds per request
MIN_REQUEST_INTERVAL = 4.5  # seconds between requests (≈13 RPM, safely under 15 RPM)

# ── API Key ──
# Set your Google AI Studio API key as an environment variable:
#   export GOOGLE_API_KEY="your-api-key-here"
# Or paste it directly below (not recommended for shared notebooks)
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY not set. Get a free key from https://aistudio.google.com/apikey \n"
        "and set it via: export GOOGLE_API_KEY='your-key'"
    )

# ── Initialize Gemini Client ──
client = genai.Client(api_key=GOOGLE_API_KEY)

# ── Logging ──
logging.basicConfig(level=logging.WARNING, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")
print(f"Model  : {MODEL_NAME}  |  Workers: {MAX_WORKERS}  |  Temp: {TEMPERATURE}  |  MaxTok: {MAX_TOKENS}")
print(f"Rate   : min {MIN_REQUEST_INTERVAL}s between requests (~{60/MIN_REQUEST_INTERVAL:.0f} RPM)")

Input  : /home/abhishek/Desktop/projects/RAG_MOLE/data/final_dataset.jsonl
Output : /home/abhishek/Desktop/projects/RAG_MOLE/data/processed/canonical_dataset_gemini.jsonl
Model  : gemini-3.1-flash-lite  |  Workers: 2  |  Temp: 0.6  |  MaxTok: 2048
Rate   : min 4.5s between requests (~13 RPM)


In [9]:
# ════════════════════════════════════════════════════════════════
# Bulletproof JSON Parser
# Handles markdown code fences, conversational filler, trailing commas
# ════════════════════════════════════════════════════════════════

def extract_and_parse_json(llm_response_string):
    # Strip markdown code fences (```json ... ```)
    cleaned = re.sub(r'```(?:json)?\s*', '', llm_response_string)
    cleaned = re.sub(r'```', '', cleaned).strip()

    # Find everything between the first { and the last }
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if not match:
        return None

    json_str = match.group(0)
    # Automatically fix trailing commas
    json_str = re.sub(r',\s*([\]}])', r'\1', json_str)

    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        return None

# Quick self-tests
assert extract_and_parse_json('Here is: {"a": 1, "b": [2,],}') == {"a": 1, "b": [2]}
assert extract_and_parse_json('```json\n{"x": 1}\n```') == {"x": 1}
assert extract_and_parse_json("") is None
print("✓ extract_and_parse_json self-tests passed")

✓ extract_and_parse_json self-tests passed


In [10]:
# ════════════════════════════════════════════════════════════════
# LLM System Prompt
# ════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are an AI Dataset Transformation Engine. Convert the input execution trace into an abstract "Canonical Plan" for a hierarchical RAG routing architecture.

PRIMITIVE TAXONOMY (primitive_id: 0-3)
- 0: READ -> Querying, fetching, searching state/data.
- 1: WRITE -> Creating, updating, modifying, executing state changes.
- 2: CHECK -> Verifying status, testing conditions, validating constraints.
- 3: PROCESS -> Computing, analyzing, transforming, formatting.

STRICT CANONICAL INTENT RULES:
1. Length: 4 to 8 words maximum.
2. Format: Abstract Action-Object imperative (e.g., "Reserve target hotel accommodation").
3. NEVER leak specific parameter values (e.g., "New York", "London").
4. NEVER copy exact function names (e.g., do NOT output "play_movie_by_title" as the intent). Use generic terms.

GROUND TRUTH TOOL RULE:
If the step uses a tool, output the exact tool_name from the input. Do NOT append "API" or "Service" to the name. If it is an internal reasoning step without a tool, output null.

OUTPUT FORMAT:
Output VALID RAW JSON ONLY. Do NOT wrap output in Markdown. Output ONLY these fields:
{
  "canonical_plan": [
    {
      "step_id": 1,
      "canonical_intent": "<Abstract intent>",
      "primitive": "<Read|Write|Check|Process>",
      "primitive_id": <0-3>,
      "domain": "<Travel|Finance|Healthcare|Productivity|DeviceControl|SoftwareEngineering|DataScience|CloudOps|InformationRetrieval|Cybersecurity>",
      "ground_truth_tool": "<tool name or null>"
    }
  ]
}

CRITICAL JSON SYNTAX RULES:
- You MUST use double quotes for all property names and string values. Do NOT use single quotes.
- Do NOT include trailing commas at the end of objects or arrays.
- Start your response immediately with the '{' character. Do NOT say "Here is the plan" or include any conversational text whatsoever.
"""

print(f"System prompt loaded ({len(SYSTEM_PROMPT)} chars)")

System prompt loaded (1821 chars)


In [11]:
# ════════════════════════════════════════════════════════════════
# Gemini 2.5 Flash LLM Caller (Google AI Studio Free Tier)
# ════════════════════════════════════════════════════════════════

import threading

# Thread-safe rate limiter: ensures MIN_REQUEST_INTERVAL between requests
_rate_lock = threading.Lock()
_last_request_time = 0.0


def _rate_limit():
    """Block until MIN_REQUEST_INTERVAL seconds have passed since the last request."""
    global _last_request_time
    with _rate_lock:
        now = time.time()
        elapsed = now - _last_request_time
        if elapsed < MIN_REQUEST_INTERVAL:
            time.sleep(MIN_REQUEST_INTERVAL - elapsed)
        _last_request_time = time.time()


def call_gemini(task_query: str, decomposed_steps: List[str]) -> Optional[Dict[str, Any]]:
    """Call Gemini 2.5 Flash via Google AI Studio and return parsed JSON, or None on failure."""

    user_prompt = (
        f"Task Query: {task_query}\n"
        f"Decomposed Steps:\n"
        + "\n".join(f"  - {s}" for s in decomposed_steps)
    )

    # Respect free-tier rate limits
    _rate_limit()

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=TEMPERATURE,
                max_output_tokens=MAX_TOKENS,
            ),
        )

        content = response.text or ""
        return extract_and_parse_json(content)
    except Exception as e:
        logger.warning(f"LLM call failed: {e}")
        return None

print("✓ call_gemini defined")

✓ call_gemini defined


In [12]:
# ════════════════════════════════════════════════════════════════
# Deterministic Record Assembly
# ════════════════════════════════════════════════════════════════

def _parse_raw_definition(raw_def: str) -> Dict[str, Any]:
    """Parse a stringified JSON raw_definition, returning {} on failure."""
    try:
        return json.loads(raw_def)
    except (json.JSONDecodeError, TypeError):
        return {}


def _clean_tool_name(name) -> Optional[str]:
    """Strip hallucinated ' API' suffix; return None for empty/null."""
    if name is None:
        return None
    cleaned = str(name).replace(" API", "").strip()
    return cleaned if cleaned and cleaned.lower() != "null" else None


def assemble_canonical_record(
    raw_input: Dict[str, Any],
    llm_output: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Deterministically combine the original input row with the LLM's
    semantic extraction into the final canonical record.
    """

    # ── 1. Sample ID ──
    sample_id = str(uuid.uuid4())

    # ── 2. Query ──
    query = raw_input["task_query"]

    # ── 3. Canonical plan + cleaning ──
    canonical_plan = llm_output.get("canonical_plan", [])

    for step in canonical_plan:
        # Capitalize primitive string
        step["primitive"] = str(step.get("primitive", "")).capitalize()

        # Clean ground_truth_tool (strip " API")
        step["ground_truth_tool"] = _clean_tool_name(step.get("ground_truth_tool"))

        # Inject requires_api flag
        step["requires_api"] = step["ground_truth_tool"] is not None

    # ── 5. Retrieval contexts (mathematical split) ──
    candidate_schemas = [
        {
            "tool_name": s["tool_name"],
            "definition": _parse_raw_definition(s["raw_definition"]),
        }
        for s in raw_input.get("candidate_tool_schemas", [])
    ]

    distractor_schemas = [
        {
            "tool_name": s["tool_name"],
            "definition": _parse_raw_definition(s["raw_definition"]),
        }
        for s in raw_input.get("distractor_tool_schemas", [])
    ]

    n_distractors = len(distractor_schemas)
    half_cut = math.ceil(n_distractors * 0.5)

    retrieval_contexts = {
        "clean":     candidate_schemas,
        "noisy_50":  candidate_schemas + distractor_schemas[:half_cut],
        "noisy_100": candidate_schemas + distractor_schemas,
    }

    return {
        "sample_id": sample_id,
        "query": query,
        "canonical_plan": canonical_plan,
        "retrieval_contexts": retrieval_contexts,
    }

print("✓ assemble_canonical_record defined")

✓ assemble_canonical_record defined


In [13]:
# ════════════════════════════════════════════════════════════════
# Per-Row Worker (called by thread pool)
# ════════════════════════════════════════════════════════════════

def process_row(line: str, line_idx: int) -> Optional[Dict[str, Any]]:
    """Process a single JSONL line: parse → LLM call → assemble."""
    try:
        raw_input = json.loads(line)
    except json.JSONDecodeError:
        logger.warning(f"Row {line_idx}: malformed JSON input, skipping.")
        return None

    task_query = raw_input.get("task_query", "")
    decomposed_steps = raw_input.get("decomposed_steps", [])

    if not task_query or not decomposed_steps:
        logger.warning(f"Row {line_idx}: missing task_query or decomposed_steps, skipping.")
        return None

    # Call Gemini 2.5 Flash
    llm_output = call_gemini(task_query, decomposed_steps)

    if llm_output is None:
        logger.warning(f"Row {line_idx}: LLM returned unparseable output, skipping.")
        return None

    # Assemble the canonical record
    try:
        record = assemble_canonical_record(raw_input, llm_output)
        return record
    except Exception as e:
        logger.warning(f"Row {line_idx}: assembly failed — {e}")
        return None

print("✓ process_row defined")

✓ process_row defined


## 🧪 Test Run — 10 Samples

Run this cell to verify the full pipeline end-to-end **before** launching the main batch loop.

In [14]:
# ════════════════════════════════════════════════════════════════
# TEST RUN — Process 10 samples with verbose output
# ════════════════════════════════════════════════════════════════

TEST_N = 10

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    test_lines = [ln.strip() for ln in [f.readline() for _ in range(TEST_N)] if ln.strip()]

print(f"Testing with {len(test_lines)} samples...\n")

test_results = []
for idx, line in enumerate(test_lines):
    raw = json.loads(line)
    print("=" * 60)
    print(f"ROW {idx}: {raw['task_query'][:80]}")
    print(f"  Steps: {len(raw['decomposed_steps'])}  |  Candidates: {len(raw['candidate_tool_schemas'])}  |  Distractors: {len(raw['distractor_tool_schemas'])}")
    print("=" * 60)

    t0 = time.time()
    record = process_row(line, idx)
    elapsed = time.time() - t0

    if record:
        test_results.append(record)
        print(f"  ✅ SUCCESS ({elapsed:.1f}s)")
        print(f"  sample_id               : {record['sample_id']}")
        print(f"  plan steps              : {len(record['canonical_plan'])}")
        for step in record["canonical_plan"]:
            tool = step["ground_truth_tool"]
            print(f"    → [{step['primitive']}] {step['canonical_intent']}  (tool={tool}, api={step['requires_api']})")
        ctx = record["retrieval_contexts"]
        print(f"  retrieval_contexts      : clean={len(ctx['clean'])}, noisy_50={len(ctx['noisy_50'])}, noisy_100={len(ctx['noisy_100'])}")
    else:
        print(f"  ❌ FAILED ({elapsed:.1f}s)")
    print()

print("═" * 60)
print(f"TEST SUMMARY: {len(test_results)}/{len(test_lines)} succeeded")
print("═" * 60)

2026-08-19 00:46:28,905 [WARNING] Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Testing with 10 samples...

ROW 0: I want to watch the movie titled 'Example Movie'
  Steps: 1  |  Candidates: 1  |  Distractors: 4
  ✅ SUCCESS (2.2s)
  sample_id               : b78ac262-eee1-48e6-96d2-be44d0ea7b01
  plan steps              : 1
    → [Write] Initiate media playback for requested title  (tool=play_movie_by_title, api=True)
  retrieval_contexts      : clean=1, noisy_50=3, noisy_100=5

ROW 1: I want to book the Hilton Hotel for December 10th, 2022
  Steps: 1  |  Candidates: 1  |  Distractors: 4
  ✅ SUCCESS (4.2s)
  sample_id               : 9214367e-b375-4b81-8b41-96a95b9ac6d6
  plan steps              : 1
    → [Write] Reserve target hotel accommodation  (tool=book_hotel, api=True)
  retrieval_contexts      : clean=1, noisy_50=3, noisy_100=5

ROW 2: I want to deliver a Birthday Gift to my friend in London, UK. Then, I need to bo
  Steps: 4  |  Candidates: 4  |  Distractors: 4
  ✅ SUCCESS (14.7s)
  sample_id               : c23ff9f1-8c6f-4ac6-a634-e2204233fc83
  plan ste

2026-08-19 00:47:59,285 [WARNING] LLM call failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
2026-08-19 00:47:59,286 [WARNING] Row 6: LLM returned unparseable output, skipping.


  ❌ FAILED (57.8s)

ROW 7: Please play the music called Moonlight Sonata.
  Steps: 1  |  Candidates: 1  |  Distractors: 4
  ✅ SUCCESS (1.8s)
  sample_id               : c1dacd56-fb3c-4d36-acac-1f9e541b705c
  plan steps              : 1
    → [Write] Execute audio media playback  (tool=play_music_by_title, api=True)
  retrieval_contexts      : clean=1, noisy_50=3, noisy_100=5

ROW 8: Make a video call to my friend with phone number +1-234-567-8910.
  Steps: 1  |  Candidates: 1  |  Distractors: 4
  ✅ SUCCESS (3.9s)
  sample_id               : a5483a93-0327-4cd9-8a10-ebe4421899cb
  plan steps              : 1
    → [Write] Initiate video communication with contact  (tool=make_video_call, api=True)
  retrieval_contexts      : clean=1, noisy_50=3, noisy_100=5

ROW 9: I want to record an audio file named 'example.wav', then organize an online meet
  Steps: 3  |  Candidates: 3  |  Distractors: 4
  ✅ SUCCESS (5.4s)
  sample_id               : d689553d-30ff-4e42-abaf-744464de3f53
  plan steps  

## 🚀 Main Execution Loop (Resumable)

This cell is **resumable** using a **progress checkpoint file** (`OUTPUT_PATH + ".progress"`).

- **Resume tracks input rows attempted** (not output lines), so skipped/failed rows never cause index drift.
- **Bounded submission window** (`MAX_WORKERS × 4`) prevents unbounded memory if early workers stall.
- **Atomic progress writes** via `os.replace()` ensure the checkpoint is never corrupted on crash.
- **Rate limiting** ensures we stay within the Gemini free-tier limit of 15 RPM.


In [ ]:
# ════════════════════════════════════════════════════════════════
# Main Execution Loop — Concurrent, Resumable & Memory-Safe
# ════════════════════════════════════════════════════════════════

PROGRESS_PATH = OUTPUT_PATH + ".progress"
MAX_BUFFER    = MAX_WORKERS * 4   # Cap in-flight futures for memory safety


def _read_progress(path: str) -> int:
    """Read the progress checkpoint; returns number of input rows already attempted."""
    try:
        with open(path, "r") as f:
            return int(f.read().strip())
    except (FileNotFoundError, ValueError):
        return 0


def _write_progress(path: str, count: int) -> None:
    """Atomically update the progress checkpoint (write-tmp + rename)."""
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(str(count))
    os.replace(tmp, path)          # atomic on POSIX


# ── Read all input lines ──
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    all_lines = [ln.strip() for ln in f if ln.strip()]

total_input = len(all_lines)

# ── Resume detection (progress-file based, NOT output line count) ──
already_done = min(_read_progress(PROGRESS_PATH), total_input)

if already_done >= total_input:
    print(f"✅ All {total_input:,} rows already processed. Nothing to do.")
    print(f"   Output: {OUTPUT_PATH}")
else:
    remaining_lines = all_lines[already_done:]
    total_remaining = len(remaining_lines)

    if already_done > 0:
        print(f"♻️  RESUMING — {already_done:,} rows attempted, {total_remaining:,} remaining")
    else:
        print(f"🚀 STARTING FRESH — {total_remaining:,} rows to process")
    print(f"Bounded submission window: {MAX_BUFFER} futures")
    print(f"Launching ThreadPoolExecutor with {MAX_WORKERS} workers…\n")

    # Safe directory creation (handles empty dirname for CWD-relative paths)
    out_dir = os.path.dirname(OUTPUT_PATH)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    success_count = 0
    fail_count    = 0
    t0 = time.time()

    # Open in append mode so we never overwrite previous progress
    with open(OUTPUT_PATH, "a", encoding="utf-8") as out_f:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

            pending_futures: Dict[Any, int] = {}        # future → local idx
            results_buffer:  Dict[int, Optional[Dict]] = {}
            next_write_idx = 0
            line_iter = iter(enumerate(remaining_lines))

            pbar = tqdm(total=total_remaining, desc="Processing", unit="row")

            # ── Seed the initial bounded batch of futures ──
            for _ in range(min(MAX_BUFFER, total_remaining)):
                idx, line = next(line_iter)
                fut = executor.submit(process_row, line, already_done + idx)
                pending_futures[fut] = idx

            # ── Main drain-and-refill loop ──
            while pending_futures:
                # Block until at least one future completes
                done, _ = wait(pending_futures, return_when=FIRST_COMPLETED)

                for future in done:
                    idx = pending_futures.pop(future)
                    try:
                        result = future.result()
                    except Exception as e:
                        logger.warning(f"Row {already_done + idx}: unexpected exception — {e}")
                        result = None

                    results_buffer[idx] = result
                    pbar.update(1)

                # Flush sequential results to disk
                flushed = False
                while next_write_idx in results_buffer:
                    rec = results_buffer.pop(next_write_idx)
                    if rec is not None:
                        out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        out_f.flush()
                        success_count += 1
                    else:
                        fail_count += 1
                    next_write_idx += 1
                    flushed = True

                # Update progress checkpoint after each flush
                if flushed:
                    _write_progress(PROGRESS_PATH, already_done + next_write_idx)

                # Refill the submission window with new tasks
                while len(pending_futures) < MAX_BUFFER:
                    try:
                        idx, line = next(line_iter)
                        fut = executor.submit(process_row, line, already_done + idx)
                        pending_futures[fut] = idx
                    except StopIteration:
                        break

            pbar.close()

    # Final progress checkpoint (covers the tail after the last flush)
    _write_progress(PROGRESS_PATH, already_done + next_write_idx)

    elapsed = time.time() - t0
    rate = success_count / elapsed if elapsed > 0 else 0

    # Count actual records in the output file for an accurate total
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        total_output_records = sum(1 for ln in f if ln.strip())

    print("\n" + "═" * 60)
    print("  GENERATION COMPLETE")
    print("═" * 60)
    print(f"  Previously attempted : {already_done:,}")
    print(f"  Processed this run   : {total_remaining:,}")
    print(f"  Successful writes    : {success_count:,}")
    print(f"  Skipped / failed     : {fail_count:,}")
    print(f"  Total output records : {total_output_records:,} / {total_input:,}")
    print(f"  Elapsed time         : {elapsed:.1f}s")
    print(f"  Throughput           : {rate:.2f} rows/s")
    print(f"  Output file          : {OUTPUT_PATH}")
    print(f"  Progress checkpoint  : {PROGRESS_PATH}")
    print("═" * 60)


In [ ]:
# ════════════════════════════════════════════════════════════════
# Sanity Check — Preview first 3 records from output
# ════════════════════════════════════════════════════════════════

if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        out_lines = f.readlines()
    print(f"Total records in output: {len(out_lines):,}\n")
    for i, line in enumerate(out_lines[:3]):
        rec = json.loads(line)
        print(f"── Record {i} ──")
        print(f"  sample_id               : {rec['sample_id']}")
        print(f"  query                   : {rec['query'][:80]}")
        print(f"  plan steps              : {len(rec['canonical_plan'])}")
        for step in rec["canonical_plan"]:
            print(f"    → [{step['primitive']}] {step['canonical_intent']}  (tool={step['ground_truth_tool']}, api={step['requires_api']})")
        ctx = rec["retrieval_contexts"]
        print(f"  retrieval_contexts      : clean={len(ctx['clean'])}, noisy_50={len(ctx['noisy_50'])}, noisy_100={len(ctx['noisy_100'])}")
        print()
else:
    print("Output file does not exist yet. Run the main loop first.")